# YT Agent — Kaggle GPU worker

Triggered by the `kaggle-dispatch.yml` GitHub Actions workflow whenever the
Vercel `/api/maintenance/needs-worker` route reports that a render is queued
and no GPU worker is alive. Boots an ephemeral FastAPI backend on a free
Kaggle T4 / P100, registers itself in Firestore, claims the queued job, runs
the pipeline, then self-terminates after 10 min of idle to preserve the
30 GPU hr/week budget.

**One-time setup**: see `kaggle/README.md`. You need to add
`GOOGLE_APPLICATION_CREDENTIALS_JSON` (and optionally R2 / SFTP) to the
Kaggle Add-ons → Secrets panel for THIS notebook.

In [ ]:
# 1) System deps + repo clone
import os, subprocess
subprocess.check_call(['apt-get', '-qq', 'update'])
subprocess.check_call(['apt-get', '-qq', 'install', '-y', 'ffmpeg'])

REPO_URL = 'https://github.com/Ahsan3301/yt_agent.git'
BRANCH   = 'main'
REPO_DIR = '/kaggle/working/yt_agent'
if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
os.chdir(REPO_DIR)
print('repo at:', os.getcwd())
print('commit:', subprocess.check_output(['git', 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 2) Python deps — base + GPU extras + browser agent
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
# Kaggle's base image already includes a CUDA torch; install the GPU
# extras NOT pinned to a specific torch wheel (skip torch line to
# avoid clobbering Kaggle's preinstalled CUDA torch).
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'decord==0.6.0', 'av==12.3.0'])
# Playwright + beautifulsoup for the NIM-driven research agent.
# `playwright install chromium` downloads the browser binary (~150 MB,
# usually <30 sec on Kaggle's fast network). Kaggle's runner has no
# system Chrome so we let Playwright manage its own.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-browser.txt'])
subprocess.check_call([sys.executable, '-m', 'playwright', 'install', 'chromium'])
import torch
print('torch:', torch.__version__, 'cuda available:', torch.cuda.is_available())
print('playwright + chromium ready — NIM research agent will activate on fact_research channels')

In [ ]:
# 3) BOOTSTRAP secrets — pulled from Kaggle's Secrets sidebar.
#
#    REQUIRED:
#      COOLIFY_BASE_URL              (your dashboard URL, e.g. https://yt-agent.thyker.online)
#      PB_URL                        (= COOLIFY_BASE_URL + /pb)
#      POCKETBASE_ADMIN_EMAIL
#      POCKETBASE_ADMIN_PASSWORD
#      RENDER_TRIGGER_KEY
#      STORAGE_PROVIDERS_ENC_KEY
#
#    OPTIONAL (legacy Firestore mode):
#      GOOGLE_APPLICATION_CREDENTIALS_JSON_B64    (base64-encoded JSON —
#       Kaggle's UI truncates multi-line secrets, so base64-encode the
#       service account JSON to keep it on one line.)
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

REQUIRED = [
    'COOLIFY_BASE_URL',
    'PB_URL',
    'POCKETBASE_ADMIN_EMAIL',
    'POCKETBASE_ADMIN_PASSWORD',
    'RENDER_TRIGGER_KEY',
    'STORAGE_PROVIDERS_ENC_KEY',
]
OPTIONAL = [
    'GOOGLE_APPLICATION_CREDENTIALS_JSON_B64',
    'INSTANCE_LABEL', 'INSTANCE_TIER',
]

missing = []
for k in REQUIRED + OPTIONAL:
    try:
        v = secrets.get_secret(k)
        if v:
            os.environ[k] = v
    except Exception:
        if k in REQUIRED:
            missing.append(k)
        continue
    if k in REQUIRED and not os.environ.get(k):
        missing.append(k)

# Defaults.
os.environ.setdefault('DB_BACKEND', 'pocketbase')
os.environ.setdefault('STORAGE_BACKEND', 'registry')
os.environ.setdefault('WORKER_MODE', 'outbound_poll')
os.environ.setdefault('INSTANCE_TIER', 'gpu')
os.environ.setdefault('INSTANCE_LABEL', 'kaggle-gpu')
# Kaggle auto-shutdown: ~25 min idle then graceful exit.
os.environ.setdefault('KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS', '1500')

if missing:
    raise SystemExit('MISSING REQUIRED SECRETS: ' + ', '.join(missing))
print('Kaggle bootstrap OK | dashboard:', os.environ['COOLIFY_BASE_URL'])
print('  worker mode:', os.environ['WORKER_MODE'], '| tier:', os.environ['INSTANCE_TIER'])


In [ ]:
# 4) (Skipped in outbound-poll mode.) Legacy cloudflared tunnel cell.
import os, subprocess, time, re

if os.environ.get('WORKER_MODE', 'tunnel').lower() != 'tunnel':
    print('outbound-poll mode — no cloudflared tunnel needed.')
else:
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.check_call([
            'wget', '-q',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-O', '/usr/local/bin/cloudflared',
        ])
        subprocess.check_call(['chmod', '+x', '/usr/local/bin/cloudflared'])
    tunnel_log = '/tmp/cloudflared.log'
    subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000',
         '--logfile', tunnel_log, '--loglevel', 'info'],
        stdout=open(tunnel_log + '.stdout', 'w'),
        stderr=subprocess.STDOUT,
    )
    url = None
    for _ in range(40):
        time.sleep(0.5)
        if not os.path.exists(tunnel_log):
            continue
        with open(tunnel_log) as f:
            m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', f.read())
            if m:
                url = m.group(0)
                break
    if not url:
        raise RuntimeError('cloudflared did not produce a URL')
    os.environ['PUBLIC_BACKEND_URL'] = url
    print('Public backend URL:', url)


In [ ]:
# 5) Boot the backend (BLOCKING). idle_watchdog will os._exit(0) after
#    KAGGLE_AUTO_SHUTDOWN_AFTER_IDLE_SECONDS once the queue is empty.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'uvicorn', 'backend.server:app',
    '--host', '0.0.0.0', '--port', '8000',
    '--log-level', 'info',
])